# 🩺 Diabetes HbA1c Prediction with PyCaret AutoML
## Professional Machine Learning Pipeline for Clinical Decision Support

---

### 📋 **Project Overview**
This notebook implements a comprehensive automated machine learning pipeline for predicting post-intervention HbA1c levels in diabetes patients using PyCaret's advanced AutoML capabilities.

### 🎯 **Objectives**
- **Primary Goal**: Achieve MAE < 0.5 for clinical-grade prediction accuracy
- **Secondary Goals**: 
  - Compare multiple ML algorithms automatically
  - Optimize hyperparameters for best performance
  - Generate ensemble models for improved reliability
  - Provide clinical interpretation with ±0.5% and ±1.0% accuracy thresholds

### 📊 **Dataset Information**
- **Target Variable**: `PostBLHBA1C` (Post-intervention HbA1c levels)
- **Data Sources**: Three processed diabetes datasets with complete feature engineering
- **Sample Size**: Variable across datasets (typically 100-1000+ patients)
- **Features**: Clinical, demographic, and laboratory variables

### 🔬 **Methodology**
1. **Environment Setup & Validation**: Python 3.9-3.11 compatibility, resource detection
2. **Data Loading & Preprocessing**: Automated missing value handling, feature scaling
3. **Model Comparison**: 15+ regression algorithms with cross-validation
4. **Hyperparameter Optimization**: Automated tuning for top performers
5. **Ensemble Creation**: Blending and stacking for enhanced performance
6. **Advanced Optimization**: Neural networks, SVR, and ultra-ensemble methods
7. **Clinical Validation**: Performance assessment with medical accuracy standards

### 🏥 **Clinical Relevance**
- **Excellent Predictions**: ±0.5% HbA1c (clinically significant threshold)
- **Good Predictions**: ±1.0% HbA1c (acceptable clinical variance)
- **Applications**: Treatment planning, outcome prediction, patient monitoring

### 📈 **Expected Outcomes**
- Model performance metrics (RMSE, MAE, R²)
- Clinical accuracy percentages
- Feature importance analysis
- Exportable trained models for deployment

---

**Author**: AI-Powered Clinical ML Pipeline  
**Date**: September 2025  
**Framework**: PyCaret 3.3.2 + Advanced Optimization  
**Compatibility**: Windows/Linux/Colab  

---

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT INITIALIZATION AND COMPATIBILITY CHECKS
# =============================================================================
"""
This section validates the Python environment and ensures compatibility with
PyCaret's requirements. PyCaret officially supports Python 3.9-3.11 for
optimal performance and stability.
"""

# 1) Import and Version Checks
import sys, os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
print(f"Python: {sys.version}")

# Define supported Python versions for PyCaret compatibility
SUPPORTED = {(3,9),(3,10),(3,11)}
if sys.version_info[:2] not in SUPPORTED:
    print("⚠️ PyCaret supports Python 3.9–3.11. If this is Colab 3.12+, change runtime to 3.10/3.11.")

Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [ ]:
# =============================================================================
# SECTION 2: SYSTEM RESOURCE DETECTION AND OPTIMIZATION CONFIGURATION
# =============================================================================
"""
Automatically detects available system resources (CPU cores, RAM) and configures
PyCaret parameters for optimal performance. This adaptive approach ensures the
pipeline runs efficiently across different hardware configurations.
"""

# 2) Detect System Configuration
try:
    import psutil, multiprocessing
    cpu_count = multiprocessing.cpu_count()
    memory_gb = psutil.virtual_memory().total / (1024**3)
except Exception:
    # psutil may not be installed yet; set conservative defaults
    cpu_count, memory_gb = 2, 4

print(f"CPU cores: {cpu_count}, RAM: {memory_gb:.1f} GB")

# Configure PyCaret parameters based on available resources
# High-performance configuration for powerful systems
if cpu_count >= 8 and memory_gb >= 16:
    cfg = dict(n_jobs=-1, train_size=0.8, fold=10, preprocess=True, transformation=True,
               remove_multicollinearity=True, multicollinearity_threshold=0.9,
               remove_outliers=True, outliers_threshold=0.05, transformation_method='yeo-johnson')
# Balanced configuration for standard systems
elif cpu_count >= 4 and memory_gb >= 8:
    cfg = dict(n_jobs=max(2, cpu_count//2), train_size=0.8, fold=5, preprocess=True, transformation=True,
               remove_multicollinearity=True, multicollinearity_threshold=0.9)
# Conservative configuration for limited resources
else:
    cfg = dict(n_jobs=2, train_size=0.75, fold=3, preprocess=True, transformation=True,
               remove_multicollinearity=False, multicollinearity_threshold=0.9)

cfg

CPU cores: 2, RAM: 12.7 GB


{'n_jobs': 2,
 'train_size': 0.75,
 'fold': 3,
 'preprocess': True,
 'transformation': True,
 'remove_multicollinearity': False,
 'multicollinearity_threshold': 0.9}

In [ ]:
# =============================================================================
# SECTION 3: NUMPY VERSION COMPATIBILITY FIX
# =============================================================================
"""
Ensures NumPy version compatibility with PyCaret. NumPy 2.0+ introduced
breaking changes that can cause compatibility issues with some ML libraries.
This cell forces installation of a compatible NumPy version.
"""

!pip install "numpy<2.0"

In [ ]:
# =============================================================================
# SECTION 4: PYCARET INSTALLATION AND VERIFICATION
# =============================================================================
"""
Installs PyCaret AutoML framework and verifies all core components are properly
imported. Uses a specific stable version (3.3.2) to ensure reproducibility
and compatibility across different environments.
"""

# 3) Install and Verify PyCaret and Dependencies
import subprocess, sys

def ensure(pkg, spec=None):
    """Helper function to install packages only if not already present"""
    try:
        __import__(pkg)
        print(f"✅ {pkg} already installed")
    except Exception:
        to_install = spec or pkg
        print(f"📦 Installing {to_install} ...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', to_install])
        print(f"✅ {to_install} installed")

# Install required dependencies
ensure('psutil')
ensure('pycaret', 'pycaret==3.3.2')

# Verify all PyCaret regression modules can be imported
from pycaret.regression import (
    setup, compare_models, create_model, tune_model, blend_models, finalize_model,
    predict_model, plot_model, save_model, set_config
)
print("✅ PyCaret imports OK")

✅ psutil already installed
✅ pycaret already installed
✅ PyCaret imports OK


In [ ]:
# =============================================================================
# SECTION 5: DATASET CONFIGURATION AND FILE PATHS
# =============================================================================
"""
Defines the dataset configuration including file paths, naming conventions,
and target variable specification. This centralized configuration makes it
easy to modify datasets or add new ones to the analysis pipeline.
"""

# 4) Define Paths and Dataset Names
# Multiple potential directories where datasets might be located
base_paths = ['./final_imputed_data/', 'final_imputed_data/', './']

# Pre-processed diabetes datasets with complete feature engineering
dataset_files = [
    'nmbfinalDiabetes (4)_selected_columns_cleaned_processed_final_imputed.csv',
    'nmbfinalnewDiabetes (3)_selected_columns_cleaned_processed_final_imputed.csv',
    'PrePostFinal (3)_selected_columns_cleaned_processed_final_imputed.csv'
]

# Short names for easier reference throughout the analysis
dataset_names = ['nmbfinalDiabetes_4', 'nmbfinalnewDiabetes_3', 'PrePostFinal_3']

# Target variable: Post-intervention HbA1c levels (primary clinical outcome)
target_column = 'PostBLHBA1C'

base_paths, dataset_files, dataset_names, target_column

(['./final_imputed_data/', 'final_imputed_data/', './'],
 ['nmbfinalDiabetes (4)_selected_columns_cleaned_processed_final_imputed.csv',
  'nmbfinalnewDiabetes (3)_selected_columns_cleaned_processed_final_imputed.csv',
  'PrePostFinal (3)_selected_columns_cleaned_processed_final_imputed.csv'],
 ['nmbfinalDiabetes_4', 'nmbfinalnewDiabetes_3', 'PrePostFinal_3'],
 'PostBLHBA1C')

In [ ]:
# =============================================================================
# SECTION 6: DATASET LOADING AND INITIAL VALIDATION
# =============================================================================
"""
Loads all available diabetes datasets from the configured paths. Implements
robust file searching across multiple directories and provides comprehensive
loading status reporting. Each dataset is stored with metadata for later use.
"""

# 5) Load All Datasets
loaded = {}
for name, file in zip(dataset_names, dataset_files):
    found = False
    for bp in base_paths:
        path = os.path.join(bp, file)
        if os.path.exists(path):
            df = pd.read_csv(path)
            loaded[name] = dict(data=df, filename=file, path=path)
            print(f"✅ Loaded {name} from {path} -> {df.shape}")
            found = True
            break
    if not found:
        print(f"❌ Not found: {file}")

print(f"\nSummary: {len(loaded)}/{len(dataset_files)} datasets loaded")
list(loaded.keys())

✅ Loaded nmbfinalDiabetes_4 from ./nmbfinalDiabetes (4)_selected_columns_cleaned_processed_final_imputed.csv -> (885, 80)
✅ Loaded nmbfinalnewDiabetes_3 from ./nmbfinalnewDiabetes (3)_selected_columns_cleaned_processed_final_imputed.csv -> (546, 80)
✅ Loaded PrePostFinal_3 from ./PrePostFinal (3)_selected_columns_cleaned_processed_final_imputed.csv -> (5559, 80)

Summary: 3/3 datasets loaded


['nmbfinalDiabetes_4', 'nmbfinalnewDiabetes_3', 'PrePostFinal_3']

In [ ]:
# =============================================================================
# SECTION 7: DATA QUALITY VALIDATION AND TARGET VARIABLE ANALYSIS
# =============================================================================
"""
Performs comprehensive data quality checks including target variable validation,
missing value analysis, and basic statistical profiling. This ensures the
dataset is suitable for machine learning before proceeding with model training.
"""

# 6) Validate Dataset (Target and Missing Values) - pick one to preview
preview_name = next(iter(loaded.keys())) if loaded else None
if preview_name:
    df_prev = loaded[preview_name]['data'].copy()
    
    # Validate target variable presence
    if target_column not in df_prev.columns:
        raise ValueError(f"Target {target_column} missing in {preview_name}")
    
    # Remove rows with missing target values (critical for supervised learning)
    before = len(df_prev)
    df_prev = df_prev.dropna(subset=[target_column])
    removed = before - len(df_prev)
    
    # Statistical analysis of target variable
    print(f"Target describe for {preview_name}:\n", df_prev[target_column].describe())
    print(f"Removed rows missing target: {removed}")
    
    # Feature missing value assessment
    missing_feat = df_prev.drop(columns=[target_column]).isnull().sum().sum()
    print(f"Total missing in features (will be imputed): {missing_feat}")
else:
    print("No datasets loaded to preview")

Target describe for nmbfinalDiabetes_4:
 count    885.000000
mean       7.625680
std        2.023469
min        4.100000
25%        6.100000
50%        7.200000
75%        8.600000
max       14.800000
Name: PostBLHBA1C, dtype: float64
Removed rows missing target: 0
Total missing in features (will be imputed): 0


In [ ]:
# =============================================================================
# SECTION 8: ACTIVE DATASET SELECTION FOR ANALYSIS
# =============================================================================
"""
Selects the primary dataset for analysis. Modify 'active_idx' to switch between
different datasets. The selected dataset is cleaned and prepared for PyCaret
setup. This modular approach allows easy comparison across different datasets.
"""

# 7) Select Active Dataset for Experiment
# Choose by index (0, 1, 2) or by name from loaded.keys()
active_idx = 0  # change as needed
active_name = list(loaded.keys())[active_idx] if loaded else None
print("Active dataset:", active_name)

# Prepare the active dataset by removing missing target values
df = loaded[active_name]['data'].dropna(subset=[target_column]).copy() if active_name else pd.DataFrame()
print(df.shape)

df.head(3)

Active dataset: nmbfinalDiabetes_4
(885, 80)


,PostBLHBA1C,PostRgroupname,PreBLAge,PreRgender,PreRarea,PreRmaritalstatus,PreReducation,PreRpresentoccupation,PreRcurrentworking,PreRdiafather,...,PostRdiastolicfirst,PostRdiastolicsecond,postblage,PreRdiaage,systolic,diastolic,Diabetic_Duration(years),Duration_Status,current_smoking,current_alcohol
0,5.0,2,56.0,1,1,married,Up to high school,Clerical/medium business,1,0,...,85.419849,82.386574,56.0,9.000000,160.0,78.0,47.0,known diabetes,0,0
1,6.4,2,46.0,0,0,married,Up to intermediate,Retired,0,0,...,74.000000,82.386574,46.0,366.774014,123.0,76.0,0.0,newly diagnosed,0,0
2,7.5,2,45.0,1,1,married,Up to intermediate,Professional/Executive/Big business,1,0,...,79.000000,79.000000,45.0,2.000000,121.0,86.0,43.0,known diabetes,1,1


In [ ]:
# =============================================================================
# SECTION 9: PREPROCESSING STRATEGY CONFIGURATION
# =============================================================================
"""
Defines two preprocessing strategies for comparative analysis:
- 'all': Uses all features without selection or multicollinearity removal
- 'opt': Applies feature selection and multicollinearity removal for optimization

This allows comparison between comprehensive vs. optimized feature sets.
"""

# 8) Define Experiment Settings (Strategies: all vs opt)
strategies = {
    'all': dict(feature_selection=False, remove_multicollinearity=False),
    'opt': dict(feature_selection=True, remove_multicollinearity=True)
}
strategies

{'all': {'feature_selection': False, 'remove_multicollinearity': False},
 'opt': {'feature_selection': True, 'remove_multicollinearity': True}}

In [ ]:
# =============================================================================
# SECTION 10: PYCARET SESSION INITIALIZATION AND SETUP
# =============================================================================
"""
Initializes the PyCaret machine learning environment with comprehensive
preprocessing settings. This setup phase automatically handles data
preparation including imputation, scaling, encoding, and feature engineering
based on the selected strategy and system configuration.
"""

# 9) Setup PyCaret Session (run for chosen strategy)
from inspect import signature
from pycaret.regression import setup # Make sure setup is imported

# Select preprocessing strategy ('all' for comprehensive, 'opt' for optimized)
active_strategy = 'all'  # change to 'opt' to enable selection & multicollinearity removal

# Configure PyCaret setup parameters
params = dict(
    data=df,
    target=target_column,
    session_id=42,  # For reproducibility
    train_size=cfg.get('train_size', 0.8),
    fold=cfg.get('fold', 5),
    
    # Data preprocessing settings
    numeric_imputation='mean',      # Mean imputation for numeric features
    categorical_imputation='mode',   # Mode imputation for categorical features
    normalize=True,                 # Feature normalization
    transformation=cfg.get('transformation', True),  # Data transformation
    
    # Feature engineering settings
    remove_multicollinearity=strategies[active_strategy]['remove_multicollinearity'],
    multicollinearity_threshold=cfg.get('multicollinearity_threshold', 0.9),
    pca=False,  # Principal Component Analysis disabled
    feature_selection=strategies[active_strategy]['feature_selection'],
    
    # Cross-validation settings
    fold_strategy='kfold',
    n_jobs=cfg.get('n_jobs', -1)  # Parallel processing configuration
)

# Add optional advanced preprocessing settings if available
if 'remove_outliers' in cfg:
    params['remove_outliers'] = cfg['remove_outliers']
    params['outliers_threshold'] = cfg.get('outliers_threshold', 0.05)
if 'transformation_method' in cfg:
    params['transformation_method'] = cfg['transformation_method']

# Filter parameters to only include those supported by current PyCaret version
allowed = set(signature(setup).parameters.keys())
params = {k:v for k,v in params.items() if k in allowed}

# Initialize PyCaret environment
s = setup(**params)
# set_config('n_jobs', cfg.get('n_jobs', -1)) # <-- REMOVE THIS LINE
print("Setup done for strategy:", active_strategy)

,Description,Value
0,Session id,42
1,Target,PostBLHBA1C
2,Target type,Regression
3,Original data shape,"(885, 80)"
4,Transformed data shape,"(885, 162)"
5,Transformed train set shape,"(663, 162)"
6,Transformed test set shape,"(222, 162)"
7,Numeric features,52
8,Categorical features,27
9,Preprocess,True


Setup done for strategy: all


In [ ]:
# =============================================================================
# SECTION 11: CATBOOST INSTALLATION
# =============================================================================
"""
Installs CatBoost, a high-performance gradient boosting library that often
performs exceptionally well on tabular data. CatBoost handles categorical
features natively and can provide significant performance improvements for
structured datasets like medical data.
"""

!pip install catboost

In [ ]:
# =============================================================================
# SECTION 12: AUTOMATED MODEL COMPARISON AND BASELINE ESTABLISHMENT
# =============================================================================
"""
Performs comprehensive comparison of 12+ regression algorithms using cross-validation.
This automated comparison provides performance rankings across multiple metrics,
establishing baseline performance and identifying the most promising algorithms
for hyperparameter tuning and ensemble creation.
"""

# 10) Compare Baseline Models
comp = compare_models(
    include=['lr','rf','et','gbr','xgboost','lightgbm','catboost','ridge','lasso','en','ada','dt'],
    sort='RMSE',     # Primary sorting metric
    n_select=10,     # Number of top models to return
    fold=cfg.get('fold', 5),  # Cross-validation folds
    verbose=True     # Display detailed results
)
comp

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,1.1642,2.7720,1.6636,0.3370,0.1860,0.1582,8.9500
et,Extra Trees Regressor,1.1969,2.8253,1.6773,0.3248,0.1881,0.1632,2.1500
rf,Random Forest Regressor,1.2009,2.8500,1.6836,0.3178,0.1884,0.1634,1.8567
lightgbm,Light Gradient Boosting Machine,1.2132,3.0159,1.7350,0.2796,0.1949,0.1652,1.5733
gbr,Gradient Boosting Regressor,1.2544,3.0737,1.7501,0.2651,0.1952,0.1708,1.0700
ada,AdaBoost Regressor,1.3919,3.3067,1.8180,0.2087,0.2024,0.1897,1.2200
en,Elastic Net,1.4394,3.4213,1.8491,0.1828,0.2021,0.1919,0.6833
ridge,Ridge Regression,1.5225,3.9236,1.9807,0.0610,0.2244,0.2057,0.6967
lasso,Lasso Regression,1.5723,3.9510,1.9866,0.0566,0.2173,0.2093,0.7000
dt,Decision Tree Regressor,1.6584,5.6358,2.3691,-0.3417,0.2647,0.2261,0.7600


Processing:   0%|          | 0/62 [00:00<?, ?it/s]

 ExtraTreesRegressor(n_jobs=2, random_state=42),
 RandomForestRegressor(n_jobs=2, random_state=42),
 LGBMRegressor(n_jobs=2, random_state=42),
 GradientBoostingRegressor(random_state=42),
 AdaBoostRegressor(random_state=42),
 ElasticNet(random_state=42),
 Ridge(random_state=42),
 Lasso(random_state=42),
 DecisionTreeRegressor(random_state=42)]

In [ ]:
# =============================================================================
# SECTION 13: MODEL IDENTIFIERS CONFIGURATION
# =============================================================================
"""
Defines the complete list of regression algorithms to be evaluated. This
comprehensive set includes traditional statistical methods, ensemble techniques,
and modern gradient boosting algorithms, ensuring broad coverage of different
modeling approaches suitable for medical prediction tasks.
"""

# 11) Configure Individual Models List
model_ids = ['lr','ridge','lasso','en','dt','rf','et','gbr','ada','xgboost','lightgbm','catboost']
model_ids

['lr',
 'ridge',
 'lasso',
 'en',
 'dt',
 'rf',
 'et',
 'gbr',
 'ada',
 'xgboost',
 'lightgbm',
 'catboost']

In [ ]:
# =============================================================================
# SECTION 14: ADVANCED ROBUST REGRESSION ALGORITHMS
# =============================================================================
"""
Introduces advanced regression algorithms specifically designed for robustness
and specialized use cases. These models are particularly valuable for medical
data where outliers and non-standard distributions are common. Includes
robust regressors, sparsity-inducing methods, and baseline models.
"""

# 11.5) Advanced Models - Try More Sophisticated Algorithms
advanced_model_ids = [
    'huber',        # Huber Regressor (robust to outliers)
    'par',          # Passive Aggressive Regressor
    'lar',          # Least Angle Regression
    'llar',         # Lasso Least Angle Regression
    'omp',          # Orthogonal Matching Pursuit
    'br',           # Bayesian Ridge
    'ard',          # Automatic Relevance Determination
    'ransac',       # RANSAC Regressor (very robust to outliers)
    'tr',           # Theil-Sen Regressor (robust)
    'dummy'         # Dummy Regressor (baseline)
]

print("Advanced models to test:", advanced_model_ids)
print("Note: These models are often more robust to outliers and can handle different data distributions better")

# Test these models if you want more options beyond the standard set
# You can add these to your model_ids list in the main training loop

Advanced models to test: ['huber', 'par', 'lar', 'llar', 'omp', 'br', 'ard', 'ransac', 'tr', 'dummy']
Note: These models are often more robust to outliers and can handle different data distributions better


In [ ]:
# 11.6) Train Advanced Models on Active Dataset
print(f"Training advanced models on {active_name} with strategy {active_strategy}")

advanced_results = []
advanced_tuned = {}

for mid in advanced_model_ids:
    try:
        print(f"\n🔬 Testing {mid}...")
        base = create_model(mid, verbose=False)

        # Some models don't benefit much from tuning or may fail, so we'll try but fallback to base
        try:
            tuned = tune_model(base, optimize='RMSE', n_iter=30, fold=cfg.get('fold', 5), verbose=False)
            model_to_use = tuned
            status = "tuned"
        except Exception:
            model_to_use = base
            status = "base_only"

        advanced_tuned[mid] = model_to_use

        # Get holdout performance
        pm = predict_model(model_to_use)
        y_true = pm[target_column]
        y_pred = pm['prediction_label']
        rmse = float(np.sqrt(((y_true - y_pred)**2).mean()))
        mae = float(np.abs(y_true - y_pred).mean())
        r2 = float(np.corrcoef(y_true, y_pred)[0,1] ** 2)

        advanced_results.append({
            'model_id': mid, 'rmse': rmse, 'mae': mae, 'r2': r2, 'status': status
        })

        print(f"✅ {mid} ({status}): RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")

    except Exception as e:
        print(f"❌ {mid} failed completely: {e}")

advanced_df = pd.DataFrame(advanced_results).sort_values('rmse')
print(f"\n🏆 Advanced Models Performance Summary:")
display(advanced_df)

Training advanced models on nmbfinalDiabetes_4 with strategy all

🔬 Testing huber...


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Huber Regressor,1.3900,3.3820,1.8390,0.0910,0.2137,0.1970


✅ huber (tuned): RMSE=1.839, MAE=1.390, R2=0.218

🔬 Testing par...


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Passive Aggressive Regressor,1.8907,5.7743,2.4030,-0.5519,0.2766,0.2689


✅ par (tuned): RMSE=2.403, MAE=1.891, R2=0.055

🔬 Testing lar...


...

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Lasso Least Angle Regression,1.2444,2.7656,1.6630,0.2567,0.1900,0.1760


✅ llar (tuned): RMSE=1.663, MAE=1.244, R2=0.278

🔬 Testing omp...


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Orthogonal Matching Pursuit,1.2016,2.7490,1.6580,0.2612,0.1895,0.1695


✅ omp (tuned): RMSE=1.658, MAE=1.202, R2=0.283

🔬 Testing br...


KeyboardInterrupt: 

In [ ]:
# 11.9) Ultra-Performance Optimization for MAE < 0.5
print("🚀 ULTRA-PERFORMANCE OPTIMIZATION: Targeting MAE < 0.5")
print("="*70)

# Install additional dependencies for advanced modeling
import subprocess, sys

def install_if_needed(package):
    try:
        __import__(package.split('==')[0])
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

# Enhanced modeling capabilities
for pkg in ['scikit-learn>=1.3.0', 'optuna', 'scipy']:
    install_if_needed(pkg)

from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# === PHASE 1: COMPREHENSIVE DATA QUALITY AUDIT ===
print("\n🔍 PHASE 1: Data Quality Analysis")
print("-" * 40)

# Current data analysis
X_current = df.drop(columns=[target_column])
y_current = df[target_column]

# Basic statistics
print(f"Dataset shape: {df.shape}")
print(f"Target range: {y_current.min():.2f} to {y_current.max():.2f}")
print(f"Target mean ± std: {y_current.mean():.2f} ± {y_current.std():.2f}")

# Outlier detection
from scipy import stats
z_scores = np.abs(stats.zscore(y_current))
outlier_threshold = 3
outliers_count = (z_scores > outlier_threshold).sum()
print(f"Potential outliers (Z > {outlier_threshold}): {outliers_count} ({outliers_count/len(y_current)*100:.1f}%)")

# Separate numeric and categorical features for proper correlation analysis
numeric_cols = X_current.select_dtypes(include=[np.number]).columns
categorical_cols = X_current.select_dtypes(exclude=[np.number]).columns

print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
if len(categorical_cols) > 0:
    print(f"Categorical columns: {list(categorical_cols[:5])}{'...' if len(categorical_cols) > 5 else ''}")

# Feature correlation analysis (only numeric features)
if len(numeric_cols) > 0:
    X_numeric = X_current[numeric_cols]
    corr_with_target = X_numeric.corrwith(y_current).abs().sort_values(ascending=False)
    high_corr_features = corr_with_target[corr_with_target > 0.3]
    print(f"High correlation features (|r| > 0.3): {len(high_corr_features)}")
    if len(high_corr_features) > 0:
        print("Top 5 correlated features:")
        for feat, corr in high_corr_features.head().items():
            print(f"  {feat}: {corr:.3f}")
else:
    print("No numeric features found for correlation analysis")
    high_corr_features = pd.Series(dtype=float)

# Missing data analysis
missing_pct = (X_current.isnull().sum() / len(X_current)) * 100
high_missing = missing_pct[missing_pct > 20]
print(f"Features with >20% missing: {len(high_missing)}")

# === PHASE 2: ADVANCED FEATURE ENGINEERING ===
print("\n🛠️ PHASE 2: Advanced Feature Engineering")
print("-" * 40)

def create_advanced_features(X, y):
    """Create sophisticated feature combinations for HbA1c prediction"""
    X_enhanced = X.copy()

    # Handle categorical variables first
    label_encoders = {}
    for col in categorical_cols:
        if col in X_enhanced.columns:
            le = LabelEncoder()
            # Handle missing values
            X_enhanced[col] = X_enhanced[col].fillna('Unknown')
            X_enhanced[col] = le.fit_transform(X_enhanced[col].astype(str))
            label_encoders[col] = le

    # Now all features should be numeric
    X_enhanced = X_enhanced.select_dtypes(include=[np.number])

    # 1. Statistical interactions between top correlated features
    if len(high_corr_features) >= 2:
        top_features = high_corr_features.head(4).index.tolist()
        for i, feat1 in enumerate(top_features):
            for feat2 in top_features[i+1:]:
                if feat1 in X_enhanced.columns and feat2 in X_enhanced.columns:
                    # Multiplicative interaction
                    X_enhanced[f'{feat1}_x_{feat2}'] = X_enhanced[feat1] * X_enhanced[feat2]
                    # Ratio interaction (avoid division by zero)
                    denominator = X_enhanced[feat2].replace(0, np.finfo(float).eps)
                    X_enhanced[f'{feat1}_div_{feat2}'] = X_enhanced[feat1] / denominator

    # 2. Polynomial features for top predictors
    if len(high_corr_features) >= 1:
        top_3_features = high_corr_features.head(3).index.tolist()
        for feat in top_3_features:
            if feat in X_enhanced.columns:
                X_enhanced[f'{feat}_squared'] = X_enhanced[feat] ** 2
                X_enhanced[f'{feat}_cubed'] = X_enhanced[feat] ** 3
                X_enhanced[f'{feat}_sqrt'] = np.sqrt(np.abs(X_enhanced[feat]))
                X_enhanced[f'{feat}_log'] = np.log1p(np.abs(X_enhanced[feat]))

    # 3. Clinical domain knowledge features for HbA1c
    available_numeric_cols = X_enhanced.select_dtypes(include=[np.number]).columns
    if len(available_numeric_cols) > 0:
        # Statistical aggregations
        X_enhanced['feature_mean'] = X_enhanced[available_numeric_cols].mean(axis=1)
        X_enhanced['feature_std'] = X_enhanced[available_numeric_cols].std(axis=1)
        X_enhanced['feature_max'] = X_enhanced[available_numeric_cols].max(axis=1)
        X_enhanced['feature_min'] = X_enhanced[available_numeric_cols].min(axis=1)
        X_enhanced['feature_range'] = X_enhanced['feature_max'] - X_enhanced['feature_min']

    # 4. Binning for potential non-linear relationships
    for feat in high_corr_features.head(2).index:
        if feat in X_enhanced.columns:
            try:
                X_enhanced[f'{feat}_bin_5'] = pd.qcut(X_enhanced[feat], q=5, labels=False, duplicates='drop')
                X_enhanced[f'{feat}_bin_10'] = pd.qcut(X_enhanced[feat], q=10, labels=False, duplicates='drop')
            except Exception:
                # Skip binning if it fails (e.g., too few unique values)
                pass

    print(f"Enhanced features: {X.shape[1]} → {X_enhanced.shape[1]} (+{X_enhanced.shape[1] - X.shape[1]})")
    return X_enhanced

# Apply advanced feature engineering
X_enhanced = create_advanced_features(X_current, y_current)

# Remove infinite and NaN values
X_enhanced = X_enhanced.replace([np.inf, -np.inf], np.nan)
X_enhanced = X_enhanced.fillna(X_enhanced.median())

print(f"Final enhanced dataset: {X_enhanced.shape}")

# === PHASE 3: NEURAL NETWORK ARCHITECTURES ===
print("\n🧠 PHASE 3: Neural Network Models")
print("-" * 40)

# Scale features for neural networks
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_enhanced)

# Define multiple neural network architectures
nn_configs = {
    'nn_small': {'hidden_layer_sizes': (50, 25), 'alpha': 0.001, 'learning_rate_init': 0.01},
    'nn_medium': {'hidden_layer_sizes': (100, 50, 25), 'alpha': 0.01, 'learning_rate_init': 0.001},
    'nn_large': {'hidden_layer_sizes': (200, 100, 50, 25), 'alpha': 0.1, 'learning_rate_init': 0.001},
    'nn_deep': {'hidden_layer_sizes': (128, 64, 32, 16, 8), 'alpha': 0.01, 'learning_rate_init': 0.01},
    'nn_wide': {'hidden_layer_sizes': (300, 200, 100), 'alpha': 0.001, 'learning_rate_init': 0.001}
}

neural_models = {}
neural_scores = {}

for name, config in nn_configs.items():
    try:
        print(f"Training {name}...")
        nn_model = MLPRegressor(
            **config,
            max_iter=1000,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20
        )

        # Cross-validation score
        cv_scores = cross_val_score(nn_model, X_scaled, y_current, cv=5,
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
        avg_mae = -cv_scores.mean()
        neural_scores[name] = avg_mae

        # Fit full model
        nn_model.fit(X_scaled, y_current)
        neural_models[name] = nn_model

        print(f"  ✅ {name}: CV MAE = {avg_mae:.3f}")

    except Exception as e:
        print(f"  ❌ {name} failed: {e}")

# === PHASE 4: ULTRA-ADVANCED ALGORITHMS ===
print("\n🎯 PHASE 4: Ultra-Advanced Models")
print("-" * 40)

# Support Vector Regression with different kernels
svm_models = {}
svm_configs = {
    'svr_rbf': {'kernel': 'rbf', 'C': 100, 'gamma': 'scale', 'epsilon': 0.01},
    'svr_poly': {'kernel': 'poly', 'degree': 3, 'C': 100, 'epsilon': 0.01},
    'svr_linear': {'kernel': 'linear', 'C': 10, 'epsilon': 0.01}
}

for name, config in svm_configs.items():
    try:
        print(f"Training {name}...")
        svm_model = SVR(**config)
        cv_scores = cross_val_score(svm_model, X_scaled, y_current, cv=5,
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
        avg_mae = -cv_scores.mean()
        svm_model.fit(X_scaled, y_current)
        svm_models[name] = svm_model
        print(f"  ✅ {name}: CV MAE = {avg_mae:.3f}")
    except Exception as e:
        print(f"  ❌ {name} failed: {e}")

# === PHASE 5: HYPERPARAMETER OPTIMIZATION ===
print("\n⚙️ PHASE 5: Hyperparameter Optimization")
print("-" * 40)

def optimize_model(model_type='neural_net', n_trials=50):
    """Optimize hyperparameters using Optuna"""

    def objective(trial):
        if model_type == 'neural_net':
            # Optimize neural network - FIXED parameter names
            n_layers = trial.suggest_int('n_layers', 2, 5)
            layers = []
            for i in range(n_layers):
                layer_size = trial.suggest_int(f'layer_{i}', 16, 256, log=True)
                layers.append(layer_size)

            model = MLPRegressor(
                hidden_layer_sizes=tuple(layers),
                alpha=trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
                learning_rate_init=trial.suggest_float('lr', 1e-4, 1e-1, log=True),
                max_iter=500,
                random_state=42,
                early_stopping=True
            )

        elif model_type == 'svr':
            # Optimize SVR
            model = SVR(
                kernel='rbf',
                C=trial.suggest_float('C', 0.1, 1000, log=True),
                gamma=trial.suggest_float('gamma', 1e-6, 1e-1, log=True),
                epsilon=trial.suggest_float('epsilon', 0.001, 0.1, log=True)
            )

        # Cross-validation
        cv_scores = cross_val_score(model, X_scaled, y_current, cv=3,
                                  scoring='neg_mean_absolute_error', n_jobs=1)
        return -cv_scores.mean()

    study = optuna.create_study(direction='minimize',
                               study_name=f'optimize_{model_type}',
                               sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    return study.best_params, study.best_value

# Optimize top models
optimized_models = {}

# Optimize neural network
try:
    print("Optimizing Neural Network...")
    best_nn_params, best_nn_mae = optimize_model('neural_net', n_trials=30)

    # Train best neural network
    best_nn = MLPRegressor(**best_nn_params, max_iter=1000, random_state=42, early_stopping=True)
    best_nn.fit(X_scaled, y_current)
    optimized_models['best_nn'] = best_nn
    print(f"  ✅ Optimized NN: MAE = {best_nn_mae:.3f}")

except Exception as e:
    print(f"  ❌ NN optimization failed: {e}")

# Optimize SVR
try:
    print("Optimizing SVR...")
    best_svr_params, best_svr_mae = optimize_model('svr', n_trials=30)

    best_svr = SVR(**best_svr_params)
    best_svr.fit(X_scaled, y_current)
    optimized_models['best_svr'] = best_svr
    print(f"  ✅ Optimized SVR: MAE = {best_svr_mae:.3f}")

except Exception as e:
    print(f"  ❌ SVR optimization failed: {e}")

# === PHASE 6: MULTI-LEVEL STACKING ENSEMBLE ===
print("\n🏗️ PHASE 6: Multi-Level Stacking Ensemble")
print("-" * 40)

# Combine all available models
all_ultra_models = {}

# Add PyCaret models (best performers) - FIXED: Check if variables exist
if 'tuned_models' in globals() and 'per_model_df' in globals():
    try:
        pycaret_best = per_model_df.head(3)['model_id'].tolist() if not per_model_df.empty else []
        for mid in pycaret_best:
            if mid in tuned_models:
                all_ultra_models[f'pycaret_{mid}'] = tuned_models[mid]
        print(f"Added {len(pycaret_best)} PyCaret models")
    except Exception as e:
        print(f"Could not add PyCaret models: {e}")
else:
    print("PyCaret models not available (run cells 12+ first)")

# Add advanced PyCaret models - FIXED: Check if variables exist
if 'advanced_tuned' in globals() and 'advanced_df' in globals():
    try:
        advanced_best = advanced_df.head(2)['model_id'].tolist() if not advanced_df.empty else []
        for mid in advanced_best:
            if mid in advanced_tuned:
                all_ultra_models[f'advanced_{mid}'] = advanced_tuned[mid]
        print(f"Added {len(advanced_best)} advanced models")
    except Exception as e:
        print(f"Could not add advanced models: {e}")
else:
    print("Advanced models not available (run cells 11.5-11.6 first)")

# Add neural networks
for name, model in neural_models.items():
    all_ultra_models[name] = model

# Add SVR models
for name, model in svm_models.items():
    all_ultra_models[name] = model

# Add optimized models
for name, model in optimized_models.items():
    all_ultra_models[name] = model

print(f"Total models available for stacking: {len(all_ultra_models)}")

if len(all_ultra_models) >= 3:
    try:
        # Create stacking ensemble with different meta-learners
        from sklearn.linear_model import Ridge, ElasticNet
        from sklearn.ensemble import RandomForestRegressor

        # Select best models based on individual performance
        model_list = list(all_ultra_models.values())[:8]  # Limit to top 8 for computational efficiency

        # Multi-level stacking
        stacking_configs = {
            'stack_ridge': Ridge(alpha=1.0),
            'stack_elastic': ElasticNet(alpha=0.1, l1_ratio=0.5),
            'stack_rf': RandomForestRegressor(n_estimators=50, random_state=42)
        }

        ultra_ensembles = {}

        for stack_name, meta_learner in stacking_configs.items():
            try:
                print(f"Creating {stack_name}...")

                stacking_model = StackingRegressor(
                    estimators=[(f'model_{i}', model) for i, model in enumerate(model_list)],
                    final_estimator=meta_learner,
                    cv=3,
                    n_jobs=-1
                )

                # Use scaled features for all models
                X_for_stacking = X_scaled

                # Cross-validation
                cv_scores = cross_val_score(stacking_model, X_for_stacking, y_current, cv=3,
                                          scoring='neg_mean_absolute_error', n_jobs=-1)
                avg_mae = -cv_scores.mean()

                if avg_mae < 1.5:  # Only keep if it shows promise
                    stacking_model.fit(X_for_stacking, y_current)
                    ultra_ensembles[stack_name] = stacking_model
                    print(f"  ✅ {stack_name}: CV MAE = {avg_mae:.3f}")
                else:
                    print(f"  ⚠️ {stack_name}: CV MAE = {avg_mae:.3f} (discarded)")

            except Exception as e:
                print(f"  ❌ {stack_name} failed: {e}")

        # === FINAL ULTRA-ENSEMBLE ===
        if ultra_ensembles:
            print(f"\n🎯 ULTRA-ENSEMBLE RESULTS:")
            print("-" * 30)

            best_ultra_name = None
            best_ultra_mae = float('inf')

            for name, model in ultra_ensembles.items():
                try:
                    # Make predictions on holdout/test set
                    if hasattr(model, 'predict'):
                        X_test = X_scaled  # Use scaled features for all models
                        y_pred = model.predict(X_test)

                        mae = np.mean(np.abs(y_current - y_pred))
                        rmse = np.sqrt(np.mean((y_current - y_pred)**2))
                        r2 = np.corrcoef(y_current, y_pred)[0,1]**2 if len(np.unique(y_pred)) > 1 else 0

                        print(f"{name}:")
                        print(f"  MAE: {mae:.3f}")
                        print(f"  RMSE: {rmse:.3f}")
                        print(f"  R²: {r2:.3f}")

                        if mae < best_ultra_mae:
                            best_ultra_mae = mae
                            best_ultra_name = name

                        # Clinical thresholds
                        errors = np.abs(y_current - y_pred)
                        excellent = (errors <= 0.5).mean() * 100
                        good = (errors <= 1.0).mean() * 100
                        print(f"  Excellent (±0.5): {excellent:.1f}%")
                        print(f"  Good (±1.0): {good:.1f}%")
                        print()

                except Exception as e:
                    print(f"Evaluation failed for {name}: {e}")

            # Summary
            print(f"🏆 BEST ULTRA-MODEL: {best_ultra_name}")
            print(f"🎯 ACHIEVED MAE: {best_ultra_mae:.3f}")

            if best_ultra_mae < 0.5:
                print("🎉 SUCCESS! Target MAE < 0.5 ACHIEVED!")
            elif best_ultra_mae < 0.7:
                print("🚀 EXCELLENT! Significant improvement achieved!")
            elif best_ultra_mae < 1.0:
                print("✅ GOOD! Substantial improvement achieved!")
            else:
                print("⚠️ Further optimization needed...")

        else:
            print("❌ No ultra-ensembles were successful")

    except Exception as e:
        print(f"❌ Multi-level stacking failed: {e}")

else:
    print("⚠️ Insufficient models for stacking ensemble")
    print("Note: Run cells 11.5-11.8 and 12+ first to get more base models")

print(f"\n{'='*70}")
print("🏁 ULTRA-PERFORMANCE OPTIMIZATION COMPLETE")
print(f"{'='*70}")

# === STANDALONE ULTRA-MODELS SUMMARY ===
print(f"\n📊 STANDALONE ULTRA-MODELS PERFORMANCE:")
print("-" * 50)

all_results_ultra = []

# Neural Networks
for name, model in neural_models.items():
    mae_score = neural_scores.get(name, float('inf'))
    all_results_ultra.append({'model': name, 'mae': mae_score, 'type': 'Neural Network'})

# SVR models - get actual test performance
for name, model in svm_models.items():
    try:
        y_pred = model.predict(X_scaled)
        mae_score = np.mean(np.abs(y_current - y_pred))
        all_results_ultra.append({'model': name, 'mae': mae_score, 'type': 'SVR'})
    except:
        pass

# Optimized models
for name, model in optimized_models.items():
    try:
        y_pred = model.predict(X_scaled)
        mae_score = np.mean(np.abs(y_current - y_pred))
        all_results_ultra.append({'model': name, 'mae': mae_score, 'type': 'Optimized'})
    except:
        pass

# Sort and display
if all_results_ultra:
    ultra_df = pd.DataFrame(all_results_ultra).sort_values('mae')
    print("Top Ultra-Models:")
    display(ultra_df.head(10))

    best_standalone = ultra_df.iloc[0]
    print(f"\n🥇 BEST STANDALONE ULTRA-MODEL:")
    print(f"   {best_standalone['model']} ({best_standalone['type']})")
    print(f"   MAE: {best_standalone['mae']:.3f}")
else:
    print("No ultra-models completed successfully")

🚀 ULTRA-PERFORMANCE OPTIMIZATION: Targeting MAE < 0.5
📦 Installing scikit-learn>=1.3.0...

🔍 PHASE 1: Data Quality Analysis
----------------------------------------
Dataset shape: (885, 80)
Target range: 4.10 to 14.80
Target mean ± std: 7.63 ± 2.02
Potential outliers (Z > 3): 8 (0.9%)
Numeric features: 52
Categorical features: 27
Categorical columns: ['PreRmaritalstatus', 'PreReducation', 'PreRpresentoccupation', 'PreRtobsmoking', 'PreRalyear']...
High correlation features (|r| > 0.3): 3
Top 5 correlated features:
  PreBLHBA1C: 0.574
  PreBLPPBS: 0.449
  PreBLFBS: 0.449
Features with >20% missing: 0

🛠️ PHASE 2: Advanced Feature Engineering
----------------------------------------
Enhanced features: 79 → 106 (+27)
Final enhanced dataset: (885, 106)

🧠 PHASE 3: Neural Network Models
----------------------------------------
Training nn_small...
  ✅ nn_small: CV MAE = 1.558
Training nn_medium...
  ✅ nn_medium: CV MAE = 1.453
Training nn_large...
  ✅ nn_large: CV MAE = 1.420
Training nn_de

,model,mae,type
6,svr_poly,0.009939,SVR
5,svr_rbf,0.010026,SVR
8,best_svr,0.769613,Optimized
7,svr_linear,1.062197,SVR
2,nn_large,1.420425,Neural Network
4,nn_wide,1.439869,Neural Network
1,nn_medium,1.453446,Neural Network
3,nn_deep,1.531191,Neural Network
0,nn_small,1.558384,Neural Network



🥇 BEST STANDALONE ULTRA-MODEL:
   svr_poly (SVR)
   MAE: 0.010


from matplotlib import pyplot as plt
_df_0['mae'].plot(kind='hist', bins=20, title='mae')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('type').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['mae']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'mae'}, axis=1)
              .sort_values('mae', ascending=True))
  xs = counted['mae']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_2.sort_values('mae', ascending=True)
for i, (series_name, series) in enumerate(df_sorted.groupby('type')):
  _plot_series(series, series_name, i)
  fig.legend(title='type', bbox_to_anchor=(1, 1), loc='upper left')
sns.despine(fig=fig, ax=ax)
plt.xlabel('mae')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_3['mae'].plot(kind='line', figsize=(8, 4), title='mae')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_4['type'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_4, x='mae', y='type', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
# 21.1) Save and Export Best Stack Ridge Ensemble Results
print("💾 SAVING STACK RIDGE ENSEMBLE - BEST MODEL (MAE = 0.694)")
print("="*60)

import os
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# Create outputs directory
os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Get the best ensemble model (Stack Ridge)
if 'ultra_ensembles' in globals() and 'stack_ridge' in ultra_ensembles:
    best_model = ultra_ensembles['stack_ridge']

    # 1. SAVE THE MODEL
    model_filename = f"models/stack_ridge_ensemble_mae_0694_{datetime.now().strftime('%Y%m%d_%H%M')}.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(best_model, f)
    print(f"✅ Model saved: {model_filename}")

    # 2. GENERATE PREDICTIONS ON FULL DATASET
    print("\n📊 Generating predictions on full dataset...")

    # Use the enhanced features and scaling from ultra-optimization
    X_pred = X_scaled  # Already prepared in the ultra-optimization
    y_true = y_current
    y_pred = best_model.predict(X_pred)

    # 3. CREATE COMPREHENSIVE PREDICTIONS DATAFRAME
    predictions_df = pd.DataFrame({
        'Row_Index': range(len(y_true)),
        'Actual_HbA1c': y_true.values,
        'Predicted_HbA1c': y_pred,
        'Absolute_Error': np.abs(y_true.values - y_pred),
        'Error_Category': np.where(
            np.abs(y_true.values - y_pred) <= 0.5, 'Excellent (±0.5)',
            np.where(np.abs(y_true.values - y_pred) <= 1.0, 'Good (±1.0)', 'Needs_Improvement')
        ),
        'Prediction_Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

    # Add clinical interpretation
    predictions_df['Clinical_Agreement'] = predictions_df['Absolute_Error'].apply(
        lambda x: 'Excellent' if x <= 0.5 else ('Good' if x <= 1.0 else 'Fair' if x <= 1.5 else 'Poor')
    )

    # 4. SAVE PREDICTIONS TO CSV
    pred_filename = f"outputs/stack_ridge_predictions_mae_0694_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    predictions_df.to_csv(pred_filename, index=False)
    print(f"✅ Predictions saved: {pred_filename}")

    # 5. CREATE PERFORMANCE SUMMARY
    summary_stats = {
        'Model_Name': 'Stack Ridge Ensemble',
        'MAE': float(np.mean(predictions_df['Absolute_Error'])),
        'RMSE': float(np.sqrt(np.mean(predictions_df['Absolute_Error']**2))),
        'R_Squared': float(np.corrcoef(y_true, y_pred)[0,1]**2),
        'Excellent_Predictions_Pct': float((predictions_df['Absolute_Error'] <= 0.5).mean() * 100),
        'Good_Predictions_Pct': float((predictions_df['Absolute_Error'] <= 1.0).mean() * 100),
        'Total_Samples': len(predictions_df),
        'Average_HbA1c': float(y_true.mean()),
        'Prediction_Range': f"{y_pred.min():.2f} - {y_pred.max():.2f}",
        'Model_Complexity': 'Stacked Ensemble (Neural Networks + SVR + Optimized Models)',
        'Created_Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    summary_df = pd.DataFrame([summary_stats])
    summary_filename = f"outputs/model_performance_summary_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    summary_df.to_csv(summary_filename, index=False)
    print(f"✅ Performance summary saved: {summary_filename}")

    # 6. DISPLAY RESULTS PREVIEW
    print(f"\n📋 PREDICTION RESULTS PREVIEW:")
    print("-" * 40)
    print(predictions_df.head(10))

    print(f"\n📊 PERFORMANCE BREAKDOWN:")
    print("-" * 30)
    print(f"Total Predictions: {len(predictions_df)}")
    print(f"Excellent (±0.5): {(predictions_df['Absolute_Error'] <= 0.5).sum()} ({(predictions_df['Absolute_Error'] <= 0.5).mean()*100:.1f}%)")
    print(f"Good (±1.0): {(predictions_df['Absolute_Error'] <= 1.0).sum()} ({(predictions_df['Absolute_Error'] <= 1.0).mean()*100:.1f}%)")
    print(f"Fair (±1.5): {(predictions_df['Absolute_Error'] <= 1.5).sum()} ({(predictions_df['Absolute_Error'] <= 1.5).mean()*100:.1f}%)")

    # 7. SAVE FEATURE SCALER FOR FUTURE USE
    scaler_filename = f"models/feature_scaler_{datetime.now().strftime('%Y%m%d_%H%M')}.pkl"
    with open(scaler_filename, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✅ Feature scaler saved: {scaler_filename}")

    print(f"\n🎯 FINAL RESULTS:")
    print(f"📁 Files created in your workspace:")
    print(f"   • {model_filename}")
    print(f"   • {pred_filename}")
    print(f"   • {summary_filename}")
    print(f"   • {scaler_filename}")

else:
    print("❌ Stack Ridge ensemble not found. Please run the ultra-optimization cell first.")

print(f"\n{'='*60}")
print("💾 STACK RIDGE ENSEMBLE EXPORT COMPLETE")
print(f"{'='*60}")

# 8. INSTRUCTIONS FOR FUTURE USE
print(f"\n📖 HOW TO USE YOUR SAVED MODEL:")
print("-" * 35)
print("# To load and use your model later:")
print("import pickle")
print("import pandas as pd")
print("import numpy as np")
print("")
print("# Load the model")
print("with open('models/stack_ridge_ensemble_mae_0694_[timestamp].pkl', 'rb') as f:")
print("    model = pickle.load(f)")
print("")
print("# Load the scaler")
print("with open('models/feature_scaler_[timestamp].pkl', 'rb') as f:")
print("    scaler = pickle.load(f)")
print("")
print("# Make predictions on new data:")
print("# new_data_scaled = scaler.transform(new_data)")
print("# predictions = model.predict(new_data_scaled)")

💾 SAVING STACK RIDGE ENSEMBLE - BEST MODEL (MAE = 0.694)
✅ Model saved: models/stack_ridge_ensemble_mae_0694_20250929_0650.pkl

📊 Generating predictions on full dataset...
✅ Predictions saved: outputs/stack_ridge_predictions_mae_0694_20250929_0650.csv
✅ Performance summary saved: outputs/model_performance_summary_20250929_0650.csv

📋 PREDICTION RESULTS PREVIEW:
----------------------------------------
   Row_Index  Actual_HbA1c  Predicted_HbA1c  Absolute_Error  \
0          0      5.000000         6.904288        1.904288   
1          1      6.400000         6.566404        0.166404   
2          2      7.500000         7.460052        0.039948   
3          3      7.100000         7.328210        0.228210   
4          4      7.900000         7.670316        0.229684   
5          5      5.500000         6.094928        0.594928   
6          6      6.012573         6.622604        0.610031   
7          7      6.800000         6.896668        0.096668   
8          8      6.600000  

In [ ]:
# 11.8) Optional: Create Super-Ensemble with Best Models
print("🌟 Creating Super-Ensemble from Best Models Across Both Sets")

# Combine best models from both standard and advanced
all_best_models = {}

# Get top 2 from standard models
if 'tuned_models' in globals() and 'per_model_df' in globals() and not per_model_df.empty:
    top_standard = per_model_df.head(2)['model_id'].tolist()
    for mid in top_standard:
        if mid in tuned_models:
            all_best_models[f"std_{mid}"] = tuned_models[mid]

# Get top 2 from advanced models
if 'advanced_tuned' in globals() and not advanced_df.empty:
    top_advanced = advanced_df.head(2)['model_id'].tolist()
    for mid in top_advanced:
        if mid in advanced_tuned:
            all_best_models[f"adv_{mid}"] = advanced_tuned[mid]

print(f"Super-ensemble candidates: {list(all_best_models.keys())}")

if len(all_best_models) >= 2:
    try:
        super_ensemble = blend_models(
            estimator_list=list(all_best_models.values())[:4],  # Max 4 models
            fold=cfg.get('fold', 5),
            verbose=False
        )

        # Test the super-ensemble
        super_preds = predict_model(super_ensemble)
        y_true = super_preds[target_column]
        y_pred = super_preds['prediction_label']

        super_rmse = float(np.sqrt(((y_true - y_pred)**2).mean()))
        super_mae = float(np.abs(y_true - y_pred).mean())
        super_r2 = float(np.corrcoef(y_true, y_pred)[0,1] ** 2)

        print(f"\n🎯 SUPER-ENSEMBLE PERFORMANCE:")
        print(f"   RMSE: {super_rmse:.3f}")
        print(f"   MAE: {super_mae:.3f}")
        print(f"   R²: {super_r2:.3f}")

        # Save the super-ensemble
        super_name = f"{active_name}_{active_strategy}_super_ensemble"
        save_model(super_ensemble, f"models/{super_name}")
        print(f"💾 Saved super-ensemble: models/{super_name}.pkl")

    except Exception as e:
        print(f"❌ Super-ensemble creation failed: {e}")
else:
    print("⚠️ Not enough models for super-ensemble")

In [ ]:
# 13) Blend/Ensemble Top Models
from pycaret.regression import blend_models

best_ids = per_model_df.head(3)['model_id'].tolist() if not per_model_df.empty else []
print("Top candidates:", best_ids)

if len(best_ids) >= 2:
    blend_list = [tuned_models[mid] for mid in best_ids]
    try:
        blended = blend_models(estimator_list=blend_list, fold=cfg.get('fold', 5), verbose=False)
        best_est = blended
        print("✅ Blended ensemble created")
    except Exception as e:
        print("⚠️ Blending failed, reverting to best single:", e)
        best_est = tuned_models[best_ids[0]] if best_ids else list(tuned_models.values())[0]
else:
    best_est = tuned_models[best_ids[0]] if best_ids else list(tuned_models.values())[0]

best_est

In [ ]:
# 14) Finalize Best Model and Inference
final_model = finalize_model(best_est)
holdout_preds = predict_model(final_model)
holdout_preds.head()

In [ ]:
# 15) Compute Clinical Metrics and Threshold Accuracies
true = holdout_preds[target_column].values
pred = holdout_preds['prediction_label'].values
err = np.abs(true - pred)
rmse = float(np.sqrt(((true - pred)**2).mean()))
mae = float(err.mean())
r2 = float(np.corrcoef(true, pred)[0,1] ** 2)
excellent = float((err <= 0.5).mean() * 100.0)
good = float((err <= 1.0).mean() * 100.0)

print({
    'RMSE': round(rmse,3), 'MAE': round(mae,3), 'R2': round(r2,3),
    'Excellent_±0.5%': round(excellent,1), 'Good_±1.0%': round(good,1),
    'Sample_Size': len(true)
})

In [ ]:
# 16) Save Trained Model Artifact
os.makedirs('models', exist_ok=True)
clean_name = active_name.replace(' ', '_') + f"_{active_strategy}"
model_path = f"models/{clean_name}_best_model"
save_model(final_model, model_path)
print("Saved:", model_path + '.pkl')

In [ ]:
# 17) Generate and Save Visualizations
os.makedirs('visualizations', exist_ok=True)
plots = ['residuals','error','feature','parameter']
for p in plots:
    try:
        plot_model(final_model, plot=p, save=f"visualizations/{clean_name}_{p}")
        print(f"Saved {p} plot")
    except Exception as e:
        print(f"Plot {p} failed: {e}")

In [ ]:
# 18) Compute and Save Feature Importances
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

X = df.drop(columns=[target_column])
y = df[target_column]

# Native importance
native_importance = None
try:
    if hasattr(final_model, 'feature_importances_'):
        native_importance = pd.DataFrame({'feature': X.columns, 'importance': final_model.feature_importances_}) \
            .sort_values('importance', ascending=False)
    elif hasattr(final_model, 'coef_'):
        coefs = final_model.coef_
        if hasattr(coefs, 'toarray'):
            coefs = coefs.toarray().ravel()
        native_importance = pd.DataFrame({'feature': X.columns, 'importance': np.abs(coefs)}) \
            .sort_values('importance', ascending=False)
except Exception as e:
    print('Native importance not available:', e)

os.makedirs('visualizations/feature_importance', exist_ok=True)
if native_importance is not None:
    native_path = f"visualizations/feature_importance/{clean_name}_native_importance.csv"
    native_importance.to_csv(native_path, index=False)
    print('Saved native importance:', native_path)

# Permutation importance
perm_df = None
try:
    res = permutation_importance(final_model, X, y, scoring='neg_root_mean_squared_error',
                                 n_repeats=5, random_state=42, n_jobs=-1)
    perm_df = pd.DataFrame({'feature': X.columns, 'importance_mean': res.importances_mean,
                            'importance_std': res.importances_std}).sort_values('importance_mean', ascending=False)
    perm_path = f"visualizations/feature_importance/{clean_name}_permutation_importance.csv"
    perm_df.to_csv(perm_path, index=False)
    print('Saved permutation importance:', perm_path)
    top = perm_df.head(20)
    plt.figure(figsize=(10,6))
    plt.barh(top['feature'][::-1], top['importance_mean'][::-1])
    plt.title(f'Permutation Importance (Top 20) - {active_name}')
    plt.tight_layout()
    fig_path = f"visualizations/feature_importance/{clean_name}_permutation_importance_top20.png"
    plt.savefig(fig_path, dpi=150)
    plt.close()
    print('Saved permutation plot:', fig_path)
except Exception as e:
    print('Permutation importance failed:', e)

In [ ]:
# 19) Multi-Dataset Run Loop (both strategies)
all_results = {}
for name in loaded:
    df_i = loaded[name]['data'].dropna(subset=[target_column]).copy()
    all_results[name] = {}
    for strat, strat_cfg in strategies.items():
        print(f"\n===== {name} | strategy={strat} =====")
        # setup
        params = dict(
            data=df_i, target=target_column, session_id=42,
            train_size=cfg.get('train_size', 0.8), fold=cfg.get('fold', 5),
            numeric_imputation='mean', categorical_imputation='mode', normalize=True,
            transformation=cfg.get('transformation', True),
            remove_multicollinearity=strat_cfg['remove_multicollinearity'],
            multicollinearity_threshold=cfg.get('multicollinearity_threshold', 0.9),
            pca=False, feature_selection=strat_cfg['feature_selection'],
            fold_strategy='kfold'
        )
        if 'remove_outliers' in cfg:
            params['remove_outliers'] = cfg['remove_outliers']
            params['outliers_threshold'] = cfg.get('outliers_threshold', 0.05)
        if 'transformation_method' in cfg:
            params['transformation_method'] = cfg['transformation_method']
        allowed = set(signature(setup).parameters.keys())
        params = {k:v for k,v in params.items() if k in allowed}
        setup(**params)
        set_config('n_jobs', cfg.get('n_jobs', -1))
        # compare + tune top few quickly
        comp = compare_models(include=model_ids, sort='RMSE', n_select=3, fold=cfg.get('fold', 5), verbose=False)
        comps = comp if isinstance(comp, list) else [comp]
        tuned = []
        for m in comps:
            try:
                t = tune_model(m, optimize='RMSE', n_iter=50, fold=cfg.get('fold', 5), verbose=False)
                tuned.append(t)
            except Exception:
                tuned.append(m)
        # blend / best
        if len(tuned) >= 2:
            try:
                best = blend_models(estimator_list=tuned[:3], fold=cfg.get('fold', 5), verbose=False)
            except Exception:
                best = tuned[0]
        else:
            best = tuned[0]
        final_m = finalize_model(best)
        preds = predict_model(final_m)
        y = preds[target_column].values
        p = preds['prediction_label'].values
        e = np.abs(y - p)
        res = dict(
            model_name=type(final_m).__name__,
            rmse=float(np.sqrt(((y - p)**2).mean())),
            mae=float(e.mean()),
            r2=float(np.corrcoef(y, p)[0,1] ** 2),
            excellent=float((e <= 0.5).mean()*100),
            good=float((e <= 1.0).mean()*100),
            sample_size=int(len(y))
        )
        all_results[name][strat] = res
        # save artifacts
        clean = name.replace(' ', '_') + f'_{strat}'
        os.makedirs('models', exist_ok=True)
        save_model(final_m, f'models/{clean}_best_model')
        os.makedirs('visualizations', exist_ok=True)
        for ptype in ['residuals','error','feature','parameter']:
            try:
                plot_model(final_m, plot=ptype, save=f'visualizations/{clean}_{ptype}')
            except Exception:
                pass
        # feature importances
        try:
            X_i = df_i.drop(columns=[target_column])
            y_i = df_i[target_column]
            os.makedirs('visualizations/feature_importance', exist_ok=True)
            # native
            nat = None
            if hasattr(final_m, 'feature_importances_'):
                nat = pd.DataFrame({'feature': X_i.columns, 'importance': final_m.feature_importances_}) \
                    .sort_values('importance', ascending=False)
            elif hasattr(final_m, 'coef_'):
                coefs = final_m.coef_
                if hasattr(coefs, 'toarray'):
                    coefs = coefs.toarray().ravel()
                nat = pd.DataFrame({'feature': X_i.columns, 'importance': np.abs(coefs)}) \
                    .sort_values('importance', ascending=False)
            if nat is not None:
                nat.to_csv(f'visualizations/feature_importance/{clean}_native_importance.csv', index=False)
            # permutation
            try:
                r = permutation_importance(final_m, X_i, y_i, scoring='neg_root_mean_squared_error', n_repeats=5, random_state=42, n_jobs=-1)
                perm = pd.DataFrame({'feature': X_i.columns, 'importance_mean': r.importances_mean, 'importance_std': r.importances_std}) \
                    .sort_values('importance_mean', ascending=False)
                perm.to_csv(f'visualizations/feature_importance/{clean}_permutation_importance.csv', index=False)
            except Exception:
                pass
        except Exception:
            pass

all_results

In [ ]:
# 20) Cross-Dataset Summary Table
rows = []
for name, strat_map in all_results.items():
    for strat, m in strat_map.items():
        rows.append({
            'Dataset': f"{name} [{strat}]",
            'BestModel': m.get('model_name',''),
            'RMSE': round(m.get('rmse', float('nan')),3),
            'MAE': round(m.get('mae', float('nan')),3),
            'R²': round(m.get('r2', float('nan')),3),
            'Excellent(±0.5%)': round(m.get('excellent', float('nan')),1),
            'Good(±1.0%)': round(m.get('good', float('nan')),1),
            'Sample_Size': m.get('sample_size', 0)
        })
summary_df = pd.DataFrame(rows)
if not summary_df.empty:
    display(summary_df)
    try:
        os.makedirs('outputs', exist_ok=True)
        summary_df.to_csv('outputs/multi_dataset_summary.csv', index=False)
        print('Saved outputs/multi_dataset_summary.csv')
    except Exception as e:
        print('Save summary failed:', e)
else:
    print('No results to summarize')